# ViT-5 (patch 4) → PanNuke cell classification (standalone)

Self-contained linear-probe / kNN evaluation of a **ViT-5 `DinoViT5` backbone trained at patch size 4** on **cell-centric 64×64 PanNuke crops**, with the data pulled straight from the **Hugging Face Hub** (`RationAI/PanNuke`).

* **Data** (§3): the three PanNuke folds are downloaded with `datasets`, and one **64×64 crop centred on each nucleus's mask centroid** is written to disk as `<CELL_CROP_ROOT>/Fold{1,2,3}_cell_patches/<Class>/*.png` (same layout as the old pre-cut crops), plus a per-fold `manifest.csv`. This runs once; later runs reuse the crops.
* **Model** (§4): the ViT-5 architecture from `vit5_arch.py` + `DinoViT5` is defined inline — RMSNorm, QK-norm, 2D RoPE on patch tokens, separate RoPE on the 4 register tokens, LayerScale (`gamma_1/2`), absolute `pos_embed` without a CLS slot, optional Gated Attention. Hyper-parameters (patch size, width, depth, heads, registers, pos-embed grid, gating) are **read from the checkpoint tensor shapes**, and loading is strict apart from the deterministic RoPE `freqs` buffers.
* **Tokens:** `[CLS] + 16×16 patch tokens + 4 registers` for a 64×64 input at patch 4. The checkpoint's `pos_embed` grid (56×56 when the model was built with `get_vit5_backbone(patch_size=4)`) is bicubically resized to 16×16, exactly like `DinoViT5._interpolate_pos_encoding`.
* **Features:** final-norm CLS token (default) or CLS ‖ mean patch token.
* **Splits:** Fold1 = train, Fold2 = val, Fold3 = test.
* **Classifiers:** logistic regression (`C = 1/l2`, `l2 = 100/(M·C)`, lbfgs) and cosine kNN.
* **Visualizations:** CLS attention maps per head and per block (§10) and PCA of patch tokens, per image and joint (§11), ported from the standalone attention / PCA scripts and adapted to ViT-5's token layout.

Requirements: `torch`, `torchvision`, `datasets`, `numpy`, `pillow`, `scikit-learn`, `matplotlib`, `tqdm`. No `timm`, `einops`, `flash-attn` or `xformers` needed (attention runs through `F.scaled_dot_product_attention`).

## 1. Configuration — edit these

In [ ]:
from pathlib import Path

# ---- checkpoint ---------------------------------------------------------------
# Any format handled by _normalize_state_dict in your ViT-5 loader:
#   {"teacher": {"backbone.*": ...}}  (do_test() teacher_checkpoint.pth)
#   {"model": {"teacher.backbone.*": ...}} / {"student": {...}} / flat official release
CHECKPOINT = Path("/tia/phd-data/u5552013/ImmunoDINO_baselines_comparison/imunocto_eval/"
                  "vit5_p4/eval/training_23999/teacher_checkpoint.pth")
MODEL_TAG = "vit5_small_p4"          # used in output file names
EXPECTED_PATCH_SIZE = 4              # sanity check against the checkpoint (None to skip)

# RoPE reference grid. NOT stored in the checkpoint, so it must match training.
# get_vit5_backbone() always builds at 224px / patch 16 and PI-resizes the patch
# afterwards, so RoPE keeps pt_seq_len = 224 // 16 = 14. If your training run built
# the model directly at patch 4, use (train_img_size // 4) instead.
ROPE_PT_SEQ_LEN = 224 // 16
REG_THETA = 100                      # only used if the checkpoint lacks rope_reg.freqs

# ---- PanNuke from the Hugging Face Hub ------------------------------------------
HF_DATASET_ID = "RationAI/PanNuke"   # splits fold1/fold2/fold3; image, instances, categories, tissue
HF_CACHE_DIR  = None                 # e.g. "/tia/phd-data/.../hf_cache"; None = ~/.cache/huggingface

# ---- cell-centric crops ----------------------------------------------------------
CELL_CROP_ROOT   = Path("./pannuke_hf_cell_crops")   # Fold{1,2,3}_cell_patches/<Class>/*.png
CROP_SIZE        = 64
CROP_PAD_MODE    = "reflect"         # nuclei near the tile edge: "reflect" or "constant"
CROP_PAD_VALUE   = 255               # fill for "constant" (255 = white background)
MIN_NUCLEUS_AREA = 10                # px; drop edge-truncated slivers (0 = keep every nucleus)
CROP_NUM_WORKERS = 8                 # processes for crop writing (0 = run in the main process)
OVERWRITE_CROPS  = False             # True = re-cut even if a fold's manifest already exists

TRAIN_FOLDS = ["Fold1"]
VAL_FOLDS   = ["Fold2"]
TEST_FOLDS  = ["Fold3"]

# ---- evaluation ------------------------------------------------------------------
OUTPUT_DIR     = Path(f"./{MODEL_TAG}_pannuke_results")
CACHE_FEATURES = True

IMG_SIZE     = 64                    # model input (crops are already 64x64)
FEATURE_MODE = "cls"                 # "cls" or "cls+patchmean"
NORM_MEAN    = (0.485, 0.456, 0.406) # match the normalisation used during ViT-5 training
NORM_STD     = (0.229, 0.224, 0.225)
BATCH_SIZE   = 256
NUM_WORKERS  = 4
USE_AMP      = True
AMP_DTYPE    = "float16"             # or "bfloat16"
SEED         = 42

LOGREG_MAX_ITER = 1000
KNN_K           = 20

## 2. Imports & setup

In [ ]:
import csv
import math
import multiprocessing as mp
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, cohen_kappa_score, log_loss,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device,
      f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CELL_CROP_ROOT.mkdir(parents=True, exist_ok=True)

## 3. PanNuke from Hugging Face → cell-centric 64×64 crops

Each HF row is one 256×256 tile with a list of binary nucleus masks (`instances`) and one label per mask (`categories`). For every nucleus:

1. centroid = mean (y, x) of its mask pixels (centre of mass, not bbox centre);
2. the tile is padded by `CROP_SIZE // 2` (reflect by default) so every nucleus gets a full crop, even at the tile edge;
3. a `CROP_SIZE × CROP_SIZE` window is cut with the centroid at pixel (32, 32).

File names follow the old convention `fold1_<tile:05d>_<nucleus:05d>_<Class>_<Tissue>.png`, so `name.split("_")[3]` is still the class. HF label ids (`0 Neoplastic, 1 Inflammatory, 2 Connective, 3 Dead, 4 Epithelial`) are remapped to the alphabetical `CLASSES` order used before, so metrics stay comparable to earlier runs.

In [ ]:
CLASSES = ["Connective", "Dead", "Epithelial", "Inflammatory", "Neoplastic"]   # our label order
IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

hf_folds = {}
for fold in TRAIN_FOLDS + VAL_FOLDS + TEST_FOLDS:
    hf_folds[fold] = load_dataset(HF_DATASET_ID, split=fold.lower(), cache_dir=HF_CACHE_DIR)
    print(f"{fold}: {len(hf_folds[fold])} tiles")

_any = next(iter(hf_folds.values()))
HF_CLASS_NAMES  = _any.features["categories"].feature.names     # read from the dataset, not hard-coded
HF_TISSUE_NAMES = _any.features["tissue"].names
assert sorted(HF_CLASS_NAMES) == sorted(CLASSES), HF_CLASS_NAMES
print("HF class ids:", dict(enumerate(HF_CLASS_NAMES)))


def _safe(name):
    """Tissue names like 'Head & Neck' -> 'Head-Neck' so '_'-splitting of file names still works."""
    return name.replace(" & ", "-").replace("&", "-").replace(" ", "-").replace("_", "-")


def nucleus_centroid(mask):
    ys, xs = np.nonzero(mask)
    if ys.size == 0:
        return None
    return float(ys.mean()), float(xs.mean()), int(ys.size)


def pad_tile(img, pad):
    if CROP_PAD_MODE == "reflect":
        return np.pad(img, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")
    return np.pad(img, ((pad, pad), (pad, pad), (0, 0)), mode="constant", constant_values=CROP_PAD_VALUE)


def crop_centered(padded, cy, cx, size, pad):
    """Cut a size x size window whose pixel (size//2, size//2) is the centroid (cy, cx)."""
    y0 = int(round(cy)) - size // 2 + pad
    x0 = int(round(cx)) - size // 2 + pad
    return padded[y0:y0 + size, x0:x0 + size]


_CROP_JOB = {}   # set before forking so worker processes inherit the (memory-mapped) HF dataset


def _crop_tile(tile_idx):
    fold, ds = _CROP_JOB["fold"], _CROP_JOB["ds"]
    row = ds[tile_idx]
    img = np.asarray(row["image"].convert("RGB"))
    H, W = img.shape[:2]
    pad = CROP_SIZE // 2
    padded = pad_tile(img, pad)
    tissue = _safe(HF_TISSUE_NAMES[row["tissue"]])
    out_rows = []
    for nuc_idx, (inst, cat) in enumerate(zip(row["instances"], row["categories"])):
        mask = np.asarray(inst, dtype=bool)
        c = nucleus_centroid(mask)
        if c is None or c[2] < MIN_NUCLEUS_AREA:
            continue
        cy, cx, area = c
        cls = HF_CLASS_NAMES[cat]
        name = f"{fold.lower()}_{tile_idx:05d}_{nuc_idx:05d}_{cls}_{tissue}.png"
        path = CELL_CROP_ROOT / f"{fold}_cell_patches" / cls / name
        if OVERWRITE_CROPS or not path.exists():
            Image.fromarray(np.ascontiguousarray(crop_centered(padded, cy, cx, CROP_SIZE, pad))).save(path)
        touches = bool(mask[0].any() or mask[-1].any() or mask[:, 0].any() or mask[:, -1].any())
        near_edge = min(cy, cx, H - 1 - cy, W - 1 - cx) < CROP_SIZE / 2   # crop needed padding
        out_rows.append([fold, tile_idx, nuc_idx, cls, tissue, f"{cy:.2f}", f"{cx:.2f}",
                         area, int(touches), int(near_edge), str(path)])
    return out_rows


MANIFEST_HEADER = ["fold", "tile_idx", "nucleus_idx", "class", "tissue", "cy", "cx",
                   "area_px", "touches_tile_border", "padded", "path"]


def make_cell_crops(fold):
    manifest = CELL_CROP_ROOT / f"{fold}_manifest.csv"
    if manifest.exists() and not OVERWRITE_CROPS:
        print(f"{fold}: crops already exist ({manifest}), skipping. Set OVERWRITE_CROPS=True to redo.")
        return manifest
    for cls in CLASSES:
        (CELL_CROP_ROOT / f"{fold}_cell_patches" / cls).mkdir(parents=True, exist_ok=True)
    _CROP_JOB.update(fold=fold, ds=hf_folds[fold])
    idxs = range(len(hf_folds[fold]))
    rows = []
    use_pool = CROP_NUM_WORKERS > 0 and "fork" in mp.get_all_start_methods()
    if use_pool:
        with mp.get_context("fork").Pool(CROP_NUM_WORKERS) as pool:
            for r in tqdm(pool.imap(_crop_tile, idxs, chunksize=8), total=len(idxs), desc=f"Cropping {fold}"):
                rows.extend(r)
    else:
        for i in tqdm(idxs, desc=f"Cropping {fold}"):
            rows.extend(_crop_tile(i))
    tmp = manifest.with_suffix(".tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.writer(f); w.writerow(MANIFEST_HEADER); w.writerows(rows)
    tmp.rename(manifest)   # manifest only appears once the whole fold finished
    counts = {c: sum(r[3] == c for r in rows) for c in CLASSES}
    print(f"{fold}: {len(rows)} nuclei -> {counts}")
    return manifest


t0 = time.time()
for fold in hf_folds:
    make_cell_crops(fold)
print(f"crop creation done in {time.time() - t0:.0f}s")

In [ ]:
# Sanity check: one HF tile with every nucleus centroid (coloured by class) and a few crop windows.
CLASS_COLORS = dict(zip(CLASSES, ["tab:green", "black", "tab:orange", "tab:blue", "tab:red"]))
ds0 = hf_folds[TRAIN_FOLDS[0]]
row = ds0[random.Random(SEED).randrange(len(ds0))]
img = np.asarray(row["image"].convert("RGB"))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(img); axes[1].imshow(img)
for j, (inst, cat) in enumerate(zip(row["instances"], row["categories"])):
    c = nucleus_centroid(np.asarray(inst, dtype=bool))
    if c is None:
        continue
    cls = HF_CLASS_NAMES[cat]
    axes[0].contour(np.asarray(inst, dtype=float), levels=[0.5], colors=[CLASS_COLORS[cls]], linewidths=0.8)
    axes[1].plot(c[1], c[0], "o", ms=3, color=CLASS_COLORS[cls])
    if j % 6 == 0:
        axes[1].add_patch(mpatches.Rectangle((round(c[1]) - CROP_SIZE // 2 - 0.5, round(c[0]) - CROP_SIZE // 2 - 0.5),
                                             CROP_SIZE, CROP_SIZE, fill=False, ec=CLASS_COLORS[cls], lw=1))
axes[0].set_title(f"instance masks — {HF_TISSUE_NAMES[row['tissue']]}", fontsize=9)
axes[1].set_title(f"centroids + some {CROP_SIZE}×{CROP_SIZE} crop windows", fontsize=9)
for ax in axes: ax.axis("off")
axes[1].legend(handles=[mpatches.Patch(color=v, label=k) for k, v in CLASS_COLORS.items()],
               fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

## 4. Model: inline ViT-5 (`DinoViT5`)

Mirrors `vit5_arch.py` (`vit_models`, `Block`, `Attention`, `RMSNorm`, `VisionRotaryEmbedding`) and the `DinoViT5` subclass from the ViT-5 loader, with identical parameter names (`patch_embed.proj`, `cls_token`, `reg_token`, `pos_embed`, `mask_token`, `blocks.N.{norm1, attn.{qkv, proj, q_norm, k_norm, rope, rope_reg, gate_proj}, norm2, mlp.{fc1, fc2}, gamma_1, gamma_2}`, `norm`).

Token layout: `[CLS, patch_1 … patch_N, reg_1 … reg_R]`. The absolute `pos_embed` is added to patch tokens **before** CLS/registers are attached (no CLS slot), RoPE is applied to patch tokens inside every attention layer, and a second RoPE (θ = `reg_theta`, 2×2 grid) is applied to the registers.

`Attention.store_attn = True` makes that layer compute softmax attention explicitly and keep it (used in §9); otherwise it runs `F.scaled_dot_product_attention`.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.variance_epsilon = eps

    def forward(self, x):
        dt = x.dtype
        x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.variance_epsilon)
        return self.weight * x.to(dt)


def rotate_half(x):
    """(x0, x1, x2, x3, ...) -> (-x1, x0, -x3, x2, ...), same as einops version in vit5_arch."""
    x1, x2 = x.unflatten(-1, (-1, 2)).unbind(-1)
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def rope_freqs(dim, theta):
    return 1.0 / (theta ** (torch.arange(0, dim, 2)[: dim // 2].float() / dim))


class VisionRotaryEmbedding(nn.Module):
    """2D axial RoPE. `freqs` is a persistent buffer (loaded from the checkpoint);
    `pt_seq_len` is not stored and must match training (ROPE_PT_SEQ_LEN)."""

    def __init__(self, dim, pt_seq_len, theta=10000):
        super().__init__()
        self.pt_seq_len = pt_seq_len
        self.register_buffer("freqs", rope_freqs(dim, theta))
        self._cache = {}

    def _cos_sin(self, n, device, dtype):
        key = (n, device, dtype)
        if key not in self._cache:
            t = torch.arange(n, device=device) / n * self.pt_seq_len
            f = torch.outer(t, self.freqs.to(device)).repeat_interleave(2, dim=-1)    # (n, d)
            f = torch.cat([f[:, None, :].expand(n, n, -1), f[None, :, :].expand(n, n, -1)], dim=-1)
            f = f.reshape(n * n, 1, -1)                                               # (n*n, 1, head_dim)
            self._cache[key] = (f.cos().to(dtype), f.sin().to(dtype))
        return self._cache[key]

    def forward(self, x):                        # x: (B, n*n, heads, head_dim)
        n = int(math.isqrt(x.shape[1]))
        cos, sin = self._cos_sin(n, x.device, x.dtype)
        return x * cos + rotate_half(x) * sin


class Attention(nn.Module):
    def __init__(self, dim, num_heads, qkv_bias, rope_pt_seq_len, rope_reg_size, num_registers,
                 reg_theta, use_rope=True, gating_type=None):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.num_registers = num_registers
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.q_norm = RMSNorm(head_dim)
        self.k_norm = RMSNorm(head_dim)
        self.rope = VisionRotaryEmbedding(head_dim // 2, rope_pt_seq_len) if use_rope else None
        self.rope_reg = (VisionRotaryEmbedding(head_dim // 2, rope_reg_size, theta=reg_theta)
                         if rope_reg_size > 0 else None)
        self.gating_type = gating_type
        if gating_type is not None:
            self.gate_proj = nn.Linear(dim, num_heads if gating_type == "headwise" else dim)
        self.store_attn, self.last_attn = False, None

    def forward(self, x):
        x_in = x
        B, N, C = x.shape
        r = N - self.num_registers
        q, k, v = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).unbind(dim=2)
        dt = q.dtype
        q, k = self.q_norm(q).to(dt), self.k_norm(k).to(dt)
        if self.rope is not None:
            q = torch.cat((q[:, :1], self.rope(q[:, 1:r]), q[:, r:]), dim=1)
            k = torch.cat((k[:, :1], self.rope(k[:, 1:r]), k[:, r:]), dim=1)
        if self.rope_reg is not None:
            q = torch.cat((q[:, :r], self.rope_reg(q[:, r:])), dim=1)
            k = torch.cat((k[:, :r], self.rope_reg(k[:, r:])), dim=1)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)       # (B, heads, N, hd)
        if self.store_attn:
            attn = ((q.float() @ k.float().transpose(-2, -1)) * self.scale).softmax(-1)
            self.last_attn = attn.detach()
            x = (attn.to(v.dtype) @ v)
        else:
            x = F.scaled_dot_product_attention(q, k, v, scale=self.scale)
        x = x.transpose(1, 2).reshape(B, N, C)
        if self.gating_type is not None:                                           # Gated Attention
            g = torch.sigmoid(self.gate_proj(x_in))
            if self.gating_type == "headwise":
                g = g.unsqueeze(-1).expand(B, N, self.num_heads, C // self.num_heads).reshape(B, N, C)
            x = x * g
        return self.proj(x)


class Mlp(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc1, self.act, self.fc2 = nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class Block(nn.Module):
    def __init__(self, dim, mlp_hidden, **attn_kw):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = Attention(dim, **attn_kw)
        self.norm2 = RMSNorm(dim)
        self.mlp = Mlp(dim, mlp_hidden)
        self.gamma_1 = nn.Parameter(1e-4 * torch.ones(dim))
        self.gamma_2 = nn.Parameter(1e-4 * torch.ones(dim))

    def forward(self, x):
        x = x + self.gamma_1 * self.attn(self.norm1(x))
        return x + self.gamma_2 * self.mlp(self.norm2(x))


class PatchEmbed(nn.Module):
    def __init__(self, patch_size, embed_dim, in_chans=3):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class ViT5(nn.Module):
    """Inference-only DinoViT5. forward() -> final-norm CLS token."""

    def __init__(self, patch_size, embed_dim, depth, num_heads, pos_grid, num_registers, mlp_hidden,
                 qkv_bias=False, rope_pt_seq_len=14, use_rope=True, reg_theta=100, gating_type=None):
        super().__init__()
        self.patch_size, self.embed_dim, self.num_registers = patch_size, embed_dim, num_registers
        self.patch_embed = PatchEmbed(patch_size, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.reg_token = nn.Parameter(torch.zeros(1, num_registers, embed_dim)) if num_registers else None
        self.pos_embed = nn.Parameter(torch.zeros(1, pos_grid * pos_grid, embed_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, embed_dim))    # unused at inference, kept for loading
        rope_reg_size = int(math.isqrt(num_registers))
        assert rope_reg_size ** 2 == num_registers, "num_registers must be a square number"
        self.blocks = nn.ModuleList([
            Block(embed_dim, mlp_hidden, num_heads=num_heads, qkv_bias=qkv_bias,
                  rope_pt_seq_len=rope_pt_seq_len, rope_reg_size=rope_reg_size, num_registers=num_registers,
                  reg_theta=reg_theta, use_rope=use_rope, gating_type=gating_type)
            for _ in range(depth)])
        self.norm = RMSNorm(embed_dim)

    def interpolate_pos_encoding(self, n_patch, h, w):
        """Same as DinoViT5._interpolate_pos_encoding (bicubic, align_corners=False, no antialias)."""
        N, dim = self.pos_embed.shape[1:]
        if n_patch == N and w == h:
            return self.pos_embed
        g = int(math.isqrt(N))
        pe = self.pos_embed.reshape(1, g, g, dim).permute(0, 3, 1, 2)
        pe = F.interpolate(pe.float(), size=(h // self.patch_size, w // self.patch_size),
                           mode="bicubic", align_corners=False).to(self.pos_embed.dtype)
        return pe.permute(0, 2, 3, 1).reshape(1, -1, dim)

    def forward_features(self, x):
        B, _, H, W = x.shape
        x = self.patch_embed(x)
        x = x + self.interpolate_pos_encoding(x.shape[1], H, W)
        x = torch.cat((self.cls_token.expand(B, -1, -1), x), dim=1)
        if self.reg_token is not None:
            x = torch.cat((x, self.reg_token.expand(B, -1, -1)), dim=1)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        R = self.num_registers
        return {"cls": x[:, 0], "patch": x[:, 1:x.shape[1] - R], "reg": x[:, x.shape[1] - R:]}

    def forward(self, x):
        return self.forward_features(x)["cls"]

In [ ]:
def _normalize_state_dict(sd):
    """Port of _normalize_state_dict from the ViT-5 loader (teacher > student, strip prefixes)."""
    for key in ("model", "state_dict"):
        if isinstance(sd, dict) and isinstance(sd.get(key), dict):
            sd = sd[key]
    if isinstance(sd.get("teacher"), dict):
        sd = sd["teacher"]
    elif isinstance(sd.get("student"), dict):
        print("no 'teacher' key -- using 'student' sub-dict (KD training checkpoint)")
        sd = sd["student"]
    for pref in ("teacher.backbone.", "student.backbone.", "backbone.", "module."):
        hits = {k[len(pref):].replace("_fsdp_wrapped_module.", ""): v for k, v in sd.items() if k.startswith(pref)}
        if hits:
            print(f"stripped prefix '{pref}' -> {len(hits)} tensors")
            sd = hits
            break
    heads = ("dino_head", "ibot_head", "gram_teacher", "contrastive_head", "head.")
    return {k: v for k, v in sd.items() if not k.startswith(heads)}


def build_vit5_from_state_dict(sd):
    """Infer every architecture hyper-parameter from tensor shapes."""
    embed_dim = sd["cls_token"].shape[-1]
    depth = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    head_dim = sd["blocks.0.attn.q_norm.weight"].shape[0]
    gate_key = "blocks.0.attn.gate_proj.weight"
    num_heads = embed_dim // head_dim
    cfg = dict(
        patch_size=sd["patch_embed.proj.weight"].shape[-1],
        embed_dim=embed_dim, depth=depth, num_heads=num_heads,
        pos_grid=int(math.isqrt(sd["pos_embed"].shape[1])),
        num_registers=sd["reg_token"].shape[1] if "reg_token" in sd else 0,
        mlp_hidden=sd["blocks.0.mlp.fc1.weight"].shape[0],
        qkv_bias="blocks.0.attn.qkv.bias" in sd,
        rope_pt_seq_len=ROPE_PT_SEQ_LEN, use_rope=True, reg_theta=REG_THETA,
        gating_type=None if gate_key not in sd else ("headwise" if sd[gate_key].shape[0] == num_heads else "elementwise"),
    )
    assert cfg["pos_grid"] ** 2 == sd["pos_embed"].shape[1], "pos_embed grid must be square"
    return ViT5(**cfg), cfg


ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = _normalize_state_dict(ckpt)
del ckpt

model, arch = build_vit5_from_state_dict(state_dict)
print("architecture:", arch)
if EXPECTED_PATCH_SIZE is not None:
    assert arch["patch_size"] == EXPECTED_PATCH_SIZE, f"checkpoint patch size is {arch['patch_size']}"

missing, unexpected = model.load_state_dict(state_dict, strict=False)
# RoPE freqs are deterministic buffers; if an older checkpoint didn't save them the defaults are exact.
missing_ok = [k for k in missing if k.endswith(("attn.rope.freqs", "attn.rope_reg.freqs"))]
missing = [k for k in missing if k not in missing_ok]
print(f"Missing keys ({len(missing)}): {missing}" + (f"  (+{len(missing_ok)} RoPE freqs recomputed)" if missing_ok else ""))
print(f"Unexpected keys ({len(unexpected)}): {unexpected}")
assert not missing and not unexpected, "Checkpoint does not match the ViT-5 architecture"

model = model.to(device).eval()
g_in = IMG_SIZE // arch["patch_size"]
print(f"Backbone params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M | "
      f"input {IMG_SIZE}px -> {g_in}x{g_in} patch tokens (+1 CLS, +{arch['num_registers']} reg); "
      f"pos_embed {arch['pos_grid']}x{arch['pos_grid']} -> {g_in}x{g_in}")

## 5. PanNuke cell-crop dataset

In [ ]:
class PanNukeCellDataset(Dataset):
    """<CELL_CROP_ROOT>/<Fold>_cell_patches/<Class>/*.png  ->  (image, label, unique_id)"""

    def __init__(self, root, folds, transform=None):
        self.transform = transform
        self.classes = CLASSES
        self.class_to_idx = {c: i for i, c in enumerate(CLASSES)}
        self.samples = []
        for fold in folds:
            for cls in CLASSES:
                d = Path(root) / f"{fold}_cell_patches" / cls
                if not d.is_dir():
                    print(f"Warning: {d} does not exist, skipping.")
                    continue
                for p in sorted(d.iterdir()):
                    if p.suffix.lower() in IMG_EXTENSIONS:
                        self.samples.append((fold, cls, p, self.class_to_idx[cls]))
        if not self.samples:
            raise RuntimeError(f"No images found for folds {folds} under {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fold, cls, path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path.stem

    def class_counts(self):
        counts = np.bincount([s[3] for s in self.samples], minlength=len(CLASSES))
        return dict(zip(CLASSES, counts.tolist()))


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),       # no-op for 64x64 crops
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

datasets = {
    "train": PanNukeCellDataset(CELL_CROP_ROOT, TRAIN_FOLDS, transform),
    "val":   PanNukeCellDataset(CELL_CROP_ROOT, VAL_FOLDS,   transform),
    "test":  PanNukeCellDataset(CELL_CROP_ROOT, TEST_FOLDS,  transform),
}
for split, ds in datasets.items():
    print(f"{split:5s} n={len(ds):6d}  {ds.class_counts()}")

In [ ]:
# A few crops per class (the nucleus should sit in the centre of every crop).
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(8, 7))
ds = datasets["train"]
by_class = {i: [j for j, s in enumerate(ds.samples) if s[3] == i] for i in range(len(CLASSES))}
for i, cls in enumerate(CLASSES):
    picks = random.sample(by_class[i], min(6, len(by_class[i])))
    for j, ax in enumerate(axes[i]):
        ax.axis("off")
        if j < len(picks):
            ax.imshow(Image.open(ds.samples[picks[j]][2]).convert("RGB"))
            ax.plot(CROP_SIZE / 2 - 0.5, CROP_SIZE / 2 - 0.5, "+", color="lime", ms=6)
    axes[i, 0].set_title(cls, fontsize=8, loc="left")
plt.tight_layout(); plt.show()

## 6. Feature extraction

In [ ]:
_amp_dtype = getattr(torch, AMP_DTYPE)


def embed(model, imgs):
    out = model.forward_features(imgs)
    if FEATURE_MODE == "cls":
        return out["cls"]
    if FEATURE_MODE == "cls+patchmean":
        return torch.cat([out["cls"], out["patch"].mean(1)], dim=-1)
    raise ValueError(FEATURE_MODE)


@torch.no_grad()
def extract_features(model, dataset, desc):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
    feats, labels, ids = [], [], []
    for imgs, y, uid in tqdm(loader, desc=desc):
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=_amp_dtype,
                            enabled=USE_AMP and device.type == "cuda"):
            f = embed(model, imgs)
        feats.append(f.float().cpu().numpy())
        labels.append(y.numpy())
        ids.extend(uid)
    return np.concatenate(feats), np.concatenate(labels), np.array(ids)


features = {}
for split, ds in datasets.items():
    cache = OUTPUT_DIR / f"{MODEL_TAG}_{FEATURE_MODE.replace('+', '_')}_{split}_features.npz"
    if CACHE_FEATURES and cache.exists():
        d = np.load(cache, allow_pickle=False)
        if len(d["labels"]) == len(ds):
            features[split] = (d["features"], d["labels"], d["ids"])
            print(f"Loaded cached {split} features {d['features'].shape} from {cache}")
            continue
        print(f"Cache {cache} has {len(d['labels'])} rows but dataset has {len(ds)} -- re-extracting")
    t0 = time.time()
    X, y, ids = extract_features(model, ds, f"Extracting {split}")
    np.savez_compressed(cache, features=X, labels=y, ids=ids)
    features[split] = (X, y, ids)
    print(f"{split}: {X.shape} in {time.time() - t0:.0f}s -> {cache}")

X_train, y_train, _ = features["train"]
X_val,   y_val,   _ = features["val"]
X_test,  y_test,  _ = features["test"]

## 7. Classifiers & metrics

In [ ]:
def compute_metrics(y_true, y_pred, proba, n_classes=len(CLASSES)):
    m = {
        "Accuracy":        accuracy_score(y_true, y_pred),
        "Balanced_Acc":    balanced_accuracy_score(y_true, y_pred),
        "Macro_F1":        f1_score(y_true, y_pred, average="macro"),
        "Weighted_F1":     f1_score(y_true, y_pred, average="weighted"),
        "Macro_Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro_Recall":    recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Cohen_Kappa":     cohen_kappa_score(y_true, y_pred),
    }
    if proba is not None:
        y_bin = label_binarize(y_true, classes=list(range(n_classes)))
        m["AUROC"] = roc_auc_score(y_bin, proba, average="macro", multi_class="ovr")
        m["mAP"]   = average_precision_score(y_bin, proba, average="macro")
        m["LogLoss"] = log_loss(y_true, proba, labels=list(range(n_classes)))
    for c, f in zip(CLASSES, f1_score(y_true, y_pred, average=None, labels=list(range(n_classes)))):
        m[f"F1_{c}"] = f
    return m


def report(name, y_true, y_pred, proba):
    m = compute_metrics(y_true, y_pred, proba)
    print(f"\n=== {name} ===")
    for k, v in m.items():
        print(f"  {k:18s} {v:.4f}")
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))
    return m

In [ ]:
# Logistic regression linear probe -- same regularisation as ablation_eval_metrics_pannuke.py
M, C = X_train.shape[1], len(np.unique(y_train))
l2 = 100 / (M * C)
print(f"features={M}, classes={C}, l2={l2:.5f}, C={1 / l2:.2f}")

t0 = time.time()
logreg = LogisticRegression(C=1 / l2, max_iter=LOGREG_MAX_ITER, solver="lbfgs")
logreg.fit(X_train, y_train)
print(f"fit in {time.time() - t0:.0f}s")

results = {}
results["logreg_val"]  = report("LogReg | val (Fold2)",  y_val,  logreg.predict(X_val),  logreg.predict_proba(X_val))
results["logreg_test"] = report("LogReg | test (Fold3)", y_test, logreg.predict(X_test), logreg.predict_proba(X_test))

In [ ]:
# kNN (cosine distance on L2-normalised features)
def l2n(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

knn = KNeighborsClassifier(n_neighbors=KNN_K, metric="cosine", n_jobs=-1)
knn.fit(l2n(X_train), y_train)
results["knn_val"]  = report(f"kNN k={KNN_K} | val (Fold2)",  y_val,  knn.predict(l2n(X_val)),  knn.predict_proba(l2n(X_val)))
results["knn_test"] = report(f"kNN k={KNN_K} | test (Fold3)", y_test, knn.predict(l2n(X_test)), knn.predict_proba(l2n(X_test)))

## 8. Summary, confusion matrix, save

In [ ]:
keys = list(results["logreg_test"].keys())
print(f"{'metric':18s}" + "".join(f"{r:>14s}" for r in results))
for k in keys:
    print(f"{k:18s}" + "".join(f"{results[r].get(k, float('nan')):14.4f}" for r in results))

out_csv = OUTPUT_DIR / f"{MODEL_TAG}_{FEATURE_MODE.replace('+', '_')}_pannuke_metrics.csv"
with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["run"] + keys)
    for r, m in results.items():
        w.writerow([r] + [m.get(k, "") for k in keys])
print("\nSaved", out_csv)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (name, clf, Xt) in zip(axes, [("LogReg", logreg, X_test), (f"kNN k={KNN_K}", knn, l2n(X_test))]):
    cm = confusion_matrix(y_test, clf.predict(Xt), normalize="true", labels=list(range(len(CLASSES))))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(f"{name} — test (Fold3), row-normalised")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=150)
plt.show()

## 9. Visualization inputs

Both visualizations below are ports of your standalone scripts (`attention_vis` with `_AttentionExtractor` and the `compute_pca` script), running on the ViT-5 model already loaded in §4, so no config file, `setup(args)` or `ssl_meta_arch_v0` import is needed.

`VIS_PATHS` can be any RGB images: crops written in §3, or your old pre-cut PanNuke crops. `None` picks one random test crop per class. Outputs go to `OUTPUT_DIR/visualizations/{attention,pca}/`.

In [ ]:
VIS_PATHS = None
# VIS_PATHS = [
#     "/tia/phd-data/u5552013/ImmunoDINO_datasets/PanNuke/Fold1_cell_patches/Connective/fold1_00002_00010_Connective_Breast.png",
#     "/tia/phd-data/u5552013/ImmunoDINO_datasets/PanNuke/Fold1_cell_patches/Epithelial/fold1_00045_00003_Epithelial_Breast.png",
# ]
ATTN_BLOCK_IDX = -1      # which transformer block to visualise (-1 = last, as in the script)
ATTN_MAX_HEADS = 8       # heads per saved grid (ViT-5 small has 6)
ATTN_ALPHA     = 0.5

VIS_DIR  = OUTPUT_DIR / "visualizations"
ATTN_DIR = VIS_DIR / "attention"
PCA_DIR  = VIS_DIR / "pca"
for d in (ATTN_DIR, PCA_DIR):
    d.mkdir(parents=True, exist_ok=True)

if VIS_PATHS is None:
    rng = random.Random(SEED)
    test_ds = datasets["test"]
    VIS_PATHS = [rng.choice([s[2] for s in test_ds.samples if s[1] == c])
                 for c in CLASSES if any(s[1] == c for s in test_ds.samples)]
VIS_PATHS = [Path(p) for p in VIS_PATHS]


def load_and_preprocess(path, img_size=IMG_SIZE):
    """Same as the attention script: square resize, /255, mean/std normalise."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    arr = (np.asarray(img).astype(np.float32) / 255.0 - np.array(NORM_MEAN)) / np.array(NORM_STD)
    return torch.from_numpy(arr).permute(2, 0, 1).float()


def unnormalize_image(t):
    """(3,H,W) normalised tensor -> (H,W,3) uint8, like _unnormalize_image."""
    mean = torch.tensor(NORM_MEAN, dtype=t.dtype, device=t.device).view(3, 1, 1)
    std = torch.tensor(NORM_STD, dtype=t.dtype, device=t.device).view(3, 1, 1)
    return ((t * std + mean).clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)


def short_label(p):
    n = Path(p).stem
    return n.split("_")[3] if n.count("_") >= 3 else n


vis_imgs = torch.stack([load_and_preprocess(p) for p in VIS_PATHS])     # (B,3,64,64)
vis_u8 = [unnormalize_image(t) for t in vis_imgs]
print("\n".join(str(p) for p in VIS_PATHS))

## 10. Attention visualization

**Why not the `qkv` hook from the script:** in ViT-5, QK-RMSNorm and RoPE are applied to q and k *after* `attn.qkv`, so recomputing `softmax(q·kᵀ)` from the raw `qkv` output gives the wrong attention. Instead the chosen block's `Attention` is switched to `store_attn=True` for one forward pass and keeps the true softmax on the normed, rotated q/k.

**Token slicing also differs:** the script uses `attn[:, :, 0, 1 + num_register_tokens:]`, which assumes DINOv2's layout `[CLS, reg…, patches]`. ViT-5 puts registers **last** (`[CLS, patches…, reg…]`), so the CLS→patch row here is `attn[:, :, 0, 1:1+N_patch]`.

Saved per image: `attn_sample_<i>_<name>.png` = `image | mean | head 0 … head H-1`, the same strip layout as `_build_attention_grid`.

In [ ]:
@torch.no_grad()
def get_attention(model, imgs, block_idx=-1):
    """(B, heads, N, N) softmax attention of one block, after QK-norm + RoPE."""
    attn_mod = model.blocks[block_idx].attn
    attn_mod.store_attn = True
    try:
        model(imgs.to(device))
        return attn_mod.last_attn.float()
    finally:
        attn_mod.store_attn, attn_mod.last_attn = False, None


def get_cls_attention_maps(attn, img_size, num_registers):
    """CLS -> patch attention, upsampled to img_size. Registers are the LAST tokens in ViT-5."""
    B, H, N, _ = attn.shape
    n_patch = N - 1 - num_registers
    g = int(math.isqrt(n_patch))
    if g * g != n_patch:
        raise ValueError(f"Token count {N} doesn't form a square patch grid ({n_patch} patches).")
    cls_attn = attn[:, :, 0, 1:1 + n_patch].reshape(B, H, g, g)
    maps = F.interpolate(cls_attn, size=(img_size, img_size), mode="bilinear", align_corners=False)
    reg_mass = attn[:, :, 0, 1 + n_patch:].sum(-1)        # (B, heads) CLS attention spent on registers
    return maps.cpu().numpy(), reg_mass.cpu().numpy()


def overlay_attention(img_u8, amap, alpha=ATTN_ALPHA):
    """JET heat-map overlay (same colormap as cv2.COLORMAP_JET, without the cv2 dependency)."""
    a = amap - amap.min()
    a = a / (a.max() + 1e-8)
    heat = (plt.cm.jet(a)[..., :3] * 255).astype(np.float32)
    return (alpha * heat + (1 - alpha) * img_u8).astype(np.uint8)


def build_attention_grid(img_u8, head_maps, alpha=ATTN_ALPHA, max_heads=ATTN_MAX_HEADS):
    panels = [img_u8, overlay_attention(img_u8, head_maps.mean(0), alpha)]
    panels += [overlay_attention(img_u8, head_maps[h], alpha) for h in range(min(len(head_maps), max_heads))]
    return Image.fromarray(np.concatenate(panels, axis=1))


attn = get_attention(model, vis_imgs, ATTN_BLOCK_IDX)
attn_maps, reg_mass = get_cls_attention_maps(attn, IMG_SIZE, model.num_registers)

for i, p in enumerate(VIS_PATHS):
    grid = build_attention_grid(vis_u8[i], attn_maps[i])
    grid.save(ATTN_DIR / f"attn_sample_{i}_{p.stem}.png")
print(f"saved {len(VIS_PATHS)} attention strips to {ATTN_DIR}")

# Inline view of the same content, plus how much CLS attention each head sends to the registers.
B, H = attn_maps.shape[:2]
fig, axes = plt.subplots(B, H + 3, figsize=(1.6 * (H + 3), 1.7 * B), squeeze=False)
for i in range(B):
    panels = [vis_u8[i], overlay_attention(vis_u8[i], attn_maps[i].mean(0))] + \
             [overlay_attention(vis_u8[i], attn_maps[i, h]) for h in range(H)]
    titles = ["image", "mean"] + [f"head {h}" for h in range(H)]
    for j, (ax, pnl) in enumerate(zip(axes[i], panels)):
        ax.imshow(pnl); ax.axis("off")
        if i == 0:
            ax.set_title(titles[j], fontsize=8)
    ax = axes[i, -1]
    ax.bar(range(H), reg_mass[i], color="gray"); ax.set_ylim(0, 1)
    ax.set_xticks(range(H)); ax.tick_params(labelsize=6)
    if i == 0:
        ax.set_title("CLS→reg mass", fontsize=8)
    axes[i, 0].text(-4, IMG_SIZE / 2, short_label(VIS_PATHS[i]), rotation=90, va="center", ha="right", fontsize=7)
blk = ATTN_BLOCK_IDX % len(model.blocks)
plt.suptitle(f"CLS attention, block {blk} (after QK-norm + RoPE)", fontsize=9)
plt.tight_layout()
plt.savefig(ATTN_DIR / f"attention_grid_block{blk}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Head-averaged CLS attention at every depth: shows where along the network the model locks onto the nucleus.
@torch.no_grad()
def all_block_cls_maps(model, imgs):
    for b in model.blocks:
        b.attn.store_attn = True
    try:
        model(imgs.to(device))
        out = [get_cls_attention_maps(b.attn.last_attn.float(), IMG_SIZE, model.num_registers)[0].mean(1)
               for b in model.blocks]                                   # depth x (B, H, W)
    finally:
        for b in model.blocks:
            b.attn.store_attn, b.attn.last_attn = False, None
    return np.stack(out, axis=1)                                        # (B, depth, H, W)


layer_maps = all_block_cls_maps(model, vis_imgs)
D = layer_maps.shape[1]
fig, axes = plt.subplots(B, D + 1, figsize=(1.2 * (D + 1), 1.3 * B), squeeze=False)
for i in range(B):
    axes[i, 0].imshow(vis_u8[i])
    axes[i, 0].set_ylabel(short_label(VIS_PATHS[i]), fontsize=6)
    for d in range(D):
        axes[i, d + 1].imshow(overlay_attention(vis_u8[i], layer_maps[i, d]))
        if i == 0:
            axes[i, d + 1].set_title(f"blk {d}", fontsize=7)
    for ax in axes[i]:
        ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Mean-over-heads CLS attention per block", fontsize=9)
plt.tight_layout()
plt.savefig(ATTN_DIR / "attention_per_block.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. PCA visualization

Port of the PCA script: the image is resized with the same `resize_transform` (height → `IMG_SIZE`, width scaled to keep aspect, both snapped to multiples of the patch size), final-norm patch tokens are taken (the equivalent of `get_intermediate_layers(n=1, reshape=True, norm=True)`; CLS and the 4 registers dropped), a 3-component whitened PCA is fitted on all patches (no foreground mask), mapped to RGB with `sigmoid(2·x)`, bicubically upsampled, and shown as `Image | PCA | Overlay`. Each figure is saved as `pca_<name>.png`.

ViT-5's RoPE (as implemented in `vit5_arch.py`) only supports a square patch grid, so a non-square image is resized to a square instead, with a printed note. PanNuke crops are square, so this doesn't trigger for them.

In [ ]:
import torchvision.transforms.functional as TF


def resize_transform(img, image_size=IMG_SIZE, patch_size=None):
    """Same as the script's resize_transform, returns an un-normalised (3,H,W) tensor in [0,1]."""
    patch_size = patch_size or model.patch_size
    w, h = img.size
    h_p = int(image_size / patch_size)
    w_p = int((w * image_size) / (h * patch_size))
    if w_p != h_p:
        print(f"  note: {w}x{h} image gives a {h_p}x{w_p} grid; ViT-5 RoPE needs a square grid -> resizing square")
        w_p = h_p
    return TF.to_tensor(TF.resize(img, (h_p * patch_size, w_p * patch_size)))


@torch.no_grad()
def patch_tokens(model, img_norm):
    """(1,3,H,W) -> (h_p*w_p, C) final-norm patch tokens."""
    return model.forward_features(img_norm.unsqueeze(0).to(device))["patch"][0].float().cpu()


def pca_rgb(proj, h_p, w_p, out_hw):
    t = torch.sigmoid(torch.as_tensor(proj, dtype=torch.float32).mul(2.0)).view(h_p, w_p, 3).permute(2, 0, 1)
    t = F.interpolate(t.unsqueeze(0), size=out_hw, mode="bicubic", align_corners=False).squeeze(0)
    return t.clamp(0, 1).permute(1, 2, 0).numpy()


pca_items = []   # (path, resized image, tokens, h_p, w_p) -- reused by the joint PCA below
for p in VIS_PATHS:
    image = Image.open(p).convert("RGB")
    img_r = resize_transform(image)
    x = patch_tokens(model, TF.normalize(img_r, mean=NORM_MEAN, std=NORM_STD))
    h_p, w_p = [d // model.patch_size for d in img_r.shape[1:]]
    pca_items.append((p, img_r, x, h_p, w_p))

    proj = PCA(n_components=3, whiten=True).fit(x.numpy()).transform(x.numpy())
    img_h, img_w = img_r.shape[1:]
    pca_up = pca_rgb(proj, h_p, w_p, (img_h, img_w))
    img_np = img_r.permute(1, 2, 0).numpy()

    with plt.rc_context({"axes.titlesize": 4}):
        fig = plt.figure(figsize=(6, 2), dpi=300)
        for k, (pnl, title) in enumerate([(image, f"Image, Size {image.size}"),
                                          (pca_up, f"PCA (3 comp), {img_h}x{img_w}"),
                                          (0.5 * img_np + 0.5 * pca_up, "Overlay")]):
            ax = fig.add_subplot(1, 3, k + 1)
            ax.imshow(pnl); ax.set_title(title); ax.axis("off")
        fig.suptitle(short_label(p), fontsize=5)
        fig.savefig(PCA_DIR / f"pca_{p.stem}.png", bbox_inches="tight")
        plt.show()
print(f"saved {len(VIS_PATHS)} PCA figures to {PCA_DIR}")

In [ ]:
# Joint PCA: one basis fitted on the patches of all images, so a colour means the same thing in every crop.

all_tok = torch.cat([it[2] for it in pca_items]).numpy()
joint = PCA(n_components=3, whiten=True).fit(all_tok)
print("explained variance ratio:", np.round(joint.explained_variance_ratio_, 3))

n = len(pca_items)
fig, axes = plt.subplots(3, n, figsize=(1.7 * n, 5.2), squeeze=False)
for i, (p, img_r, x, h_p, w_p) in enumerate(pca_items):
    img_np = img_r.permute(1, 2, 0).numpy()
    rgb = pca_rgb(joint.transform(x.numpy()), h_p, w_p, tuple(img_r.shape[1:]))
    for r, pnl in enumerate([img_np, rgb, 0.5 * img_np + 0.5 * rgb]):
        axes[r, i].imshow(pnl); axes[r, i].axis("off")
    axes[0, i].set_title(short_label(p), fontsize=7)
for r, t in enumerate(["image", "joint PCA", "overlay"]):
    axes[r, 0].text(-4, img_r.shape[1] / 2, t, rotation=90, va="center", ha="right", fontsize=7)
plt.suptitle("Joint PCA of patch tokens (shared basis)", fontsize=9)
plt.tight_layout()
plt.savefig(PCA_DIR / "pca_joint.png", dpi=200, bbox_inches="tight")
plt.show()